In [1]:
from datasets import load_dataset, load_from_disk
import polars as pl
import pandas as pd
import json
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from replay.metrics import Recall, Precision, HitRate
import faiss
from functools import reduce
import datasets
import torch
from tqdm import tqdm
import os
from datetime import datetime

NUM_PROC = 32
CACHE_DIR = "/home/jupyter/filestore/storage/"

/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [2]:
DATA_PATH = "/home/jupyter/filestore/storage/datasets/user_clicks_20230501"

dataset = load_from_disk(DATA_PATH)

In [3]:
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(-1)

polars.config.Config

In [4]:
polars_ds = dataset.to_polars()

In [5]:
index = faiss.read_index("data/item_index_tiny.faiss")

with open("data/item_ids", "rb") as fp:
    item_ids = pickle.load(fp)

In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 560.53it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
test_items = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") > TRAIN_END_DT)
    .filter(pl.col("date") <=  TEST_END_DT)
    .select("item_id")
    .unique()["item_id"]
    .to_list()
)

train_items = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= TRAIN_END_DT)
    .select("item_id")
    .unique()["item_id"]
    .to_list()
)

In [11]:
len(train_items), len(test_items), len(list(set(train_items) & set(test_items))), len(list(set(train_items) | set(test_items)))

(4308568, 2388712, 761464, 5935816)

In [12]:
761464 / 2388712

0.3187759763420622

In [5]:
TRAIN_END_DT = pd.to_datetime("2023-05-14")
TEST_END_DT = pd.to_datetime("2023-05-21")

train_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= TRAIN_END_DT)
)

all_clicks = (
    train_interactions
    .select(
        pl.col("user_id"),
        pl.col("c2_name"),
        pl.col("name"),
        pl.col("item_id"),
        pl.col("stime"),
        pl.col("stime").rank("dense", descending=True).over("user_id").alias("rn")
    )
)

In [8]:
query_clicks_5 = (
    all_clicks
    .filter(pl.col("rn") <= 5)
    .select("name", "item_id")
    .unique()
)

In [46]:
names = query_clicks_5["name"].to_list()
ids = query_clicks_5["item_id"].to_list()

In [51]:
batch_size = 4096
similar_items = {}

replace_func = np.vectorize(lambda x: item_ids[x])

for i in tqdm(range(0, len(names), batch_size)):
    cur_names = names[i:i+batch_size]
    cur_ids = ids[i:i+batch_size]
    queries = model.encode(cur_names, batch_size=batch_size, normalize_embeddings=True)
    _, idx = index.search(queries, k=20)
    recs = replace_func(idx)
    cur_similar_items = {cur_ids[i]: recs[i, :][recs[i, :] != cur_ids[i]].tolist() for i in range(len(cur_ids))}
    similar_items = {**similar_items, **cur_similar_items}

100%|██████████| 169/169 [09:09<00:00,  3.25s/it]


In [52]:
#with open("data/similar_items_lst5_top20", "wb") as fp:
#    pickle.dump(similar_items, fp)

In [6]:
with open("data/similar_items_lst5_top50", "rb") as fp:
    similar_items = pickle.load(fp)

In [14]:
def get_similar_items(row):
    return list(reduce(lambda x, y: x + y, [similar_items[item_id] for item_id in row["last_clicks"] if item_id in similar_items]))

In [19]:
user_last_clicks_5 = (
     all_clicks
    .filter(pl.col("rn") <= 5)
    .groupby("user_id")
    .agg(pl.col("item_id").alias("last_clicks"))
)

In [21]:
recs_5 = (
    user_last_clicks_5
    .with_columns(
        pl.struct(["last_clicks"]).apply(get_similar_items).alias("recs")
    )
)

In [8]:
query_clicks_80 = (
    all_clicks
    .filter(pl.col("rn") <= 80)
    .select("name", "item_id")
    .unique()
)

In [9]:
names = query_clicks_80["name"].to_list()
ids = query_clicks_80["item_id"].to_list()

In [ ]:
batch_size = 4096
similar_items = {}

replace_func = np.vectorize(lambda x: item_ids[x])

for i in tqdm(range(0, len(names), batch_size)):
    cur_names = names[i:i+batch_size]
    cur_ids = ids[i:i+batch_size]
    queries = model.encode(cur_names, batch_size=batch_size, normalize_embeddings=True)
    _, idx = index.search(queries, k=5)
    recs = replace_func(idx)
    cur_similar_items = {cur_ids[i]: recs[i, :][recs[i, :] != cur_ids[i]].tolist() for i in range(len(cur_ids))}
    similar_items = {**similar_items, **cur_similar_items}

  6%|▋         | 45/701 [02:28<35:49,  3.28s/it]

In [ ]:
with open("data/similar_items_lst80_top5", "wb") as fp:
    pickle.dump(similar_items, fp)

In [8]:
query_clicks_20 = (
    all_clicks
    .filter(pl.col("rn") <= 20)
    .select("name", "item_id")
    .unique()
)

In [9]:
names = query_clicks_20["name"].to_list()
ids = query_clicks_20["item_id"].to_list()

In [11]:
batch_size = 4096 * 2
similar_items = {}

replace_func = np.vectorize(lambda x: item_ids[x])

for i in tqdm(range(0, len(names), batch_size)):
    cur_names = names[i:i+batch_size]
    cur_ids = ids[i:i+batch_size]
    queries = model.encode(cur_names, batch_size=batch_size, normalize_embeddings=True)
    _, idx = index.search(queries, k=41)
    recs = replace_func(idx)
    cur_similar_items = {cur_ids[i]: recs[i, :][recs[i, :] != cur_ids[i]].tolist() for i in range(len(cur_ids))}
    similar_items = {**similar_items, **cur_similar_items}

100%|██████████| 196/196 [22:20<00:00,  6.84s/it]


In [44]:
with open("data/similar_items_lst20_top10", "wb") as fp:
    pickle.dump(similar_items, fp)

In [12]:
user_last_clicks_20 = (
     all_clicks
    .filter(pl.col("rn") <= 20)
    .groupby("user_id")
    .agg(pl.col("item_id").alias("last_clicks"))
)

In [7]:
with open("data/similar_items_lst20_top20", "rb") as fp:
    similar_items = pickle.load(fp)

In [18]:
recs_20 = (
    user_last_clicks_20
    .with_columns(
        pl.struct(["last_clicks"]).apply(get_similar_items).alias("recs")
    )
)

In [13]:
#recs_20.select("user_id", "item2item_recs").write_parquet("item2item_recs.parquet")

In [6]:
recs_20 = pl.read_parquet("item2item_recs.parquet").rename({"item2item_recs": "recs"})

In [7]:
test_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") > TRAIN_END_DT)
    .filter(pl.col("date") <=  TEST_END_DT)
    .groupby("user_id")
    .agg(pl.col("item_id").alias("future_clicks"))
    .join(
        recs_20,
        on="user_id",
        how="inner"
    )
)

In [19]:
recs_stats = (
    recs_20
    .with_columns(pl.col("recs").apply(len).alias("recs_count"))
    .select(
        pl.min("recs_count").alias("min_recs_count"),
        pl.max("recs_count").alias("max_recs_count"),
        pl.mean("recs_count").alias("mean_recs_count")
    )
)

recs_stats

min_recs_count,max_recs_count,mean_recs_count
i64,i64,f64
40,960,407.179547


In [24]:
TOP_K_VALUES = [10, 100, 960]

def calc_recall(row):
    return Recall._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_precision(row):
    return Precision._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_hitrate(row):
    return HitRate._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

metrics = (
    test_interactions
    .with_columns(
        pl.struct(["future_clicks", "recs"]).apply(calc_recall).alias("recall"),
        pl.struct(["future_clicks", "recs"]).apply(calc_precision).alias("precision"),
        pl.struct(["future_clicks", "recs"]).apply(calc_hitrate).alias("hitrate"),
    )
    .select(
        pl.col("recall").arr.get(0).mean().alias("recall@10"),
        pl.col("recall").arr.get(1).mean().alias("recall@100"),
        pl.col("recall").arr.get(2).mean().alias("recall@1000"),
        pl.col("precision").arr.get(0).mean().alias("precision@10"),
        pl.col("precision").arr.get(1).mean().alias("precision@100"),
        pl.col("precision").arr.get(2).mean().alias("precision@1000"),
        pl.col("hitrate").arr.get(0).mean().alias("hitrate@10"),
        pl.col("hitrate").arr.get(1).mean().alias("hitrate@100"),
        pl.col("hitrate").arr.get(2).mean().alias("hitrate@1000"),
        pl.col("hitrate").arr.get(0).sum().alias("hitrate_sum@10"), # количество рекомендаций, попавших в отложенную выборку
        pl.col("hitrate").arr.get(1).sum().alias("hitrate_sum@100"), # количество рекомендаций, попавших в отложенную выборку
        pl.col("hitrate").arr.get(2).sum().alias("hitrate_sum@1000") # количество рекомендаций, попавших в отложенную выборку
    )
    .head(5)
)

In [25]:
metrics # 40 ближайших к последним кликам

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.007301,0.031215,0.091201,0.007104,0.003746,0.001579,0.052304,0.186384,0.430713,6018.0,21445.0,49557.0


In [41]:
metrics # 20 ближайших к последним кликам

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.007292,0.035024,0.067852,0.0071,0.004236,0.002637,0.052174,0.215291,0.371847,6003.0,24771.0,42784.0
